# Problem 1: finite differences for a variable-coefficient elliptic problem

We solve the two-point boundary value problem

$$
\begin{equation}
-\frac{d}{dx}\left(a(x)\frac{du}{dx}\right) + c(x)u(x) = f(x),
\qquad 0 < x < 1,
\qquad u(0)=u(1)=0.
\end{equation}
$$

This notebook is a notebook version of `scripts/problem1_finite_difference.py`. The goal is to build a second-order finite-difference method for the elliptic operator, verify the expected convergence rate with a manufactured solution, and then solve the same boundary value problem for the forcing $f(x)=1$.

The important structural feature is that the highest-order term is in divergence form. Instead of expanding $-(a u')'$ into $-a u''-a'u'$, we discretize the flux $a u'$ directly at half-grid points. That gives a conservative stencil and naturally handles variable coefficients.


## Grid and conservative finite-difference stencil

Let $N$ be the number of subintervals, $h=1/N$, and $x_j=jh$ for $j=0,1,\ldots,N$. The unknowns are the interior values

$$
\begin{equation}
v_j \approx u(x_j), \qquad j=1,\ldots,N-1,
\end{equation}
$$

with boundary values $v_0=v_N=0$. Define the half-node coefficients

$$
\begin{equation}
a_{j+1/2}=a(x_j+h/2), \qquad a_{j-1/2}=a(x_j-h/2).
\end{equation}
$$

At the midpoint $x_{j+1/2}$, the flux is approximated by

$$
a(x_{j+1/2})u'(x_{j+1/2})
\approx
 a_{j+1/2}\frac{v_{j+1}-v_j}{h}.
$$

Therefore the discrete divergence of this flux at $x_j$ is

$$
\begin{equation}
-(a u')'(x_j)
\approx
-\frac{1}{h}\left[
 a_{j+1/2}\frac{v_{j+1}-v_j}{h}
-
 a_{j-1/2}\frac{v_j-v_{j-1}}{h}
\right].
\end{equation}
$$

After collecting coefficients, the finite-difference equation is

$$
\begin{equation}
-\frac{a_{j-1/2}}{h^2}v_{j-1}
+\left(\frac{a_{j+1/2}+a_{j-1/2}}{h^2}+c_j\right)v_j
-\frac{a_{j+1/2}}{h^2}v_{j+1}
= f_j,
\qquad j=1,\ldots,N-1.
\end{equation}
$$

For smooth $a$, $c$, and $u$, the midpoint flux approximation is second order. Hence we expect the maximum-norm error to scale like $O(h^2)$ when the solution is smooth and the elliptic problem is well-conditioned.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

plt.rcParams.update({
    "figure.figsize": (8, 4.8),
    "axes.grid": True,
    "lines.linewidth": 2,
})


def find_project_root(start=None):
    """Find the repository root whether the notebook is launched from root or notebooks/."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts").is_dir() and (candidate / "figures").is_dir():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
FIGURES_DIR = PROJECT_ROOT / "figures" / "problem1"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def relative_to_root(path):
    return Path(path).resolve().relative_to(PROJECT_ROOT)


## Banded linear system

The stencil only couples $v_{j-1}$, $v_j$, and $v_{j+1}$, so the matrix is tridiagonal. The code stores the matrix in the compact banded layout expected by `scipy.linalg.solve_banded`:

$$
\begin{equation}
A =
\begin{bmatrix}
 b_1 & u_1 \\
 \ell_2 & b_2 & u_2 \\
 & \ddots & \ddots & \ddots \\
 & & \ell_{N-1} & b_{N-1}
\end{bmatrix},
\end{equation}
$$

where

$$
\begin{equation}
b_j=\frac{a_{j+1/2}+a_{j-1/2}}{h^2}+c(x_j),
\qquad
u_j=-\frac{a_{j+1/2}}{h^2},
\qquad
\ell_j=-\frac{a_{j-1/2}}{h^2}.
\end{equation}
$$

If $a(x)>0$ and $c(x)\ge 0$, the continuous problem satisfies a maximum-principle intuition, and the discrete matrix has the same elliptic sign pattern.


In [ ]:
def solve_fd(N, a_func, c_func, f_func):
    """
    Solve -(a u')' + c u = f on (0,1) with u(0)=u(1)=0.

    Interior nodes are j = 1, ..., N-1 with h = 1/N.
    Returns the interior grid x and the finite-difference solution v.
    """
    h = 1.0 / N
    x = np.linspace(h, 1.0 - h, N - 1)

    x_plus = x + 0.5 * h
    x_minus = x - 0.5 * h
    a_plus = a_func(x_plus)
    a_minus = a_func(x_minus)

    main_diag = (a_plus + a_minus) / h**2 + c_func(x)
    upper_diag = -a_plus[:-1] / h**2
    lower_diag = -a_minus[1:] / h**2

    rhs = f_func(x)

    M = len(x)
    ab = np.zeros((3, M))
    ab[0, 1:] = upper_diag
    ab[1, :] = main_diag
    ab[2, :-1] = lower_diag

    v = solve_banded((1, 1), ab, rhs)
    return x, v


## Manufactured solution test

To verify the method, choose

$$
\begin{equation}
a(x)=1+x^2, \qquad c(x)=e^x, \qquad u(x)=x\sin(2\pi x).
\end{equation}
$$

This exact solution satisfies $u(0)=u(1)=0$. We manufacture the forcing by substituting $u$ into the differential equation:

$$
\begin{equation}
f(x)=-(a(x)u'(x))' + c(x)u(x).
\end{equation}
$$

For this particular $u$,

$$
u'(x)=\sin(2\pi x)+2\pi x\cos(2\pi x),
$$

and

$$
u''(x)=4\pi\cos(2\pi x)-4\pi^2x\sin(2\pi x).
$$

Since $a'(x)=2x$,

$$
\begin{equation}
-(a u')'= -\left(a'(x)u'(x)+a(x)u''(x)\right).
\end{equation}
$$

This gives an exact right-hand side, so any observed error is caused by the discretization and floating-point roundoff rather than by modeling uncertainty.


In [ ]:
def a(x):
    return 1.0 + x**2


def c(x):
    return np.exp(x)


def u_exact(x):
    return x * np.sin(2.0 * np.pi * x)


def u_prime(x):
    return np.sin(2.0 * np.pi * x) + 2.0 * np.pi * x * np.cos(2.0 * np.pi * x)


def d_au_prime(x):
    """Return -(a u')' for the manufactured solution."""
    ap = 2.0 * x
    up = u_prime(x)
    upp = 4.0 * np.pi * np.cos(2.0 * np.pi * x) - 4.0 * np.pi**2 * x * np.sin(2.0 * np.pi * x)
    return -(ap * up + a(x) * upp)


def f_manufactured(x):
    return d_au_prime(x) + c(x) * u_exact(x)


## Convergence study

The script measures the maximum error

$$
\begin{equation}
\|e_h\|_{\infty}=\max_{1\le j\le N-1}|v_j-u(x_j)|.
\end{equation}
$$

When $N$ doubles, $h$ is halved. A second-order method should satisfy

$$
\begin{equation}
\|e_h\|_{\infty}\approx C h^2,
\qquad
\log_2\left(\frac{\|e_h\|_\infty}{\|e_{h/2}\|_\infty}\right)\approx 2.
\end{equation}
$$


In [ ]:
Ns = [10, 20, 40, 80, 160, 320]
errors = []

for N in Ns:
    x_int, v = solve_fd(N, a, c, f_manufactured)
    err = np.max(np.abs(v - u_exact(x_int)))
    errors.append(err)
    print(f"N={N:4d}  h={1/N:.5f}  max-error={err:.3e}")

errors = np.array(errors)
Ns_arr = np.array(Ns)
rates = np.log2(errors[:-1] / errors[1:])

print("\nConvergence rates (expected to be approximately 2):")
for i, r in enumerate(rates):
    print(f"  N={Ns[i]} -> {Ns[i+1]}: rate={r:.3f}")


In [ ]:
N_plot = 160
x_int, v = solve_fd(N_plot, a, c, f_manufactured)
x_full = np.concatenate([[0.0], x_int, [1.0]])
v_full = np.concatenate([[0.0], v, [0.0]])
u_full = u_exact(x_full)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x_full, u_full, "b-", label=r"Exact $u(x)=x\sin(2\pi x)$")
axes[0].plot(x_full, v_full, "r--", label=f"FD solution, N={N_plot}")
axes[0].set_xlabel("x")
axes[0].set_title("Manufactured solution test")
axes[0].legend()

h_ref = 1.0 / Ns_arr
axes[1].loglog(h_ref, errors, "bo-", label="Max error")
axes[1].loglog(h_ref, errors[0] * (h_ref / h_ref[0])**2, "k--", label=r"$O(h^2)$ reference")
axes[1].set_xlabel(r"$h=1/N$")
axes[1].set_ylabel("Max error")
axes[1].set_title("Convergence plot")
axes[1].legend()

fig.tight_layout()
manufactured_solution_path = FIGURES_DIR / "manufactured_solution.png"
fig.savefig(manufactured_solution_path, dpi=120)
print(f"Saved: {relative_to_root(manufactured_solution_path)}")
plt.show()


## Solving the problem with $f(x)=1$

The second experiment keeps

$$
\begin{equation}
a(x)=1+x^2, \qquad c(x)=e^x,
\end{equation}
$$

but replaces the manufactured forcing by $f(x)=1$. The problem is now

$$
\begin{equation}
-((1+x^2)u')' + e^x u = 1,
\qquad 0<x<1,
\qquad u(0)=u(1)=0.
\end{equation}
$$

Because the forcing is nonnegative, $a(x)>0$, and $c(x)>0$, the elliptic maximum-principle intuition suggests that the solution should be nonnegative in the interior. The script checks this numerically after solving on a fine grid.


In [ ]:
def f_one(x):
    return np.ones_like(x)


N_compute = 400
x_int, v = solve_fd(N_compute, a, c, f_one)
x_full = np.concatenate([[0.0], x_int, [1.0]])
v_full = np.concatenate([[0.0], v, [0.0]])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_full, v_full, "b-")
ax.set_xlabel("x")
ax.set_ylabel("u(x)")
ax.set_title(r"Solution of $-(a(x)u')' + c(x)u = 1$, $a=1+x^2$, $c=e^x$")
fig.tight_layout()

f_equals_one_path = FIGURES_DIR / "f_equals_1.png"
fig.savefig(f_equals_one_path, dpi=120)
print(f"Saved: {relative_to_root(f_equals_one_path)}")
print(f"\nMin of numerical solution with f=1: {v_full.min():.6f}  (should be >= 0)")
print(f"Max of numerical solution with f=1: {v_full.max():.6f}")
plt.show()
